## Introduction

In this code, the Land Surface Temperature (LST) is computed using the thermal band (band 10) of Landsat 8 - Landsat Collection 2 Level 2 Dataset. Rather than modifying this (thermal), one can also use the values of the band as a direct measure for LST. 

This code is written in geemap. The code is largely based on the Javascript code of Ujaval Gandhi (2021, supplement) for the "Derive LST from Landsat Images'.

In this script, the Python geemap package is used. In order to convert the JavaScript (JS) code of Google Earth Engine (GEE) into geemap, we used the conversion Python command ["Geemap "geemap.js_snippet_to_py" command"](https://book.geemap.org/chapters/03_gee_data.html#converting-javascript-to-python) that is available in the geemap package. 

In this code, the water pixels are masked out. This follows the work of Chakraborty (2023). When looking at the road density per ward, it makes much sense to mask out the water pixels as otherwise incorporating the water pixels in the LST computation can distort the mean LST computation per ward.

## Importing the libraries

In [ ]:
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import pandas as pd

In [ ]:
cloud_project = 'FILL IN YOUR CLOUD PROJECT'

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

## Reading in and preparing the data

In [ ]:
# Define the paths
BASE_DIR = Path.cwd().parents[1]
print(BASE_DIR)

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [ ]:
## Detailed wards
df_inpath = PROCESSED_DIR / "bang_fix_mapshaper.shp"
print(df_inpath)

In [ ]:
# https://geemap.org/notebooks/10_shapefiles/ 
# Convert to something geemap can understand
bangalore = geemap.shp_to_ee(df_inpath)

In [ ]:
print(f'The number of wards in Bangalore is {bangalore.size().getInfo()}')

In [ ]:
bangGeometry = bangalore.union(50)

Map = geemap.Map()
Map.centerObject(bangGeometry, zoom=10)
Map # Uncomment if you want to see the map. 
## Otherwise you can wait till later Maps as all layers are in them

In [ ]:
Map.addLayer(bangGeometry, {}, 'Bangalore outline')
Map

In [ ]:
## Add the ward data
image = ee.Image().paint(bangalore, 0, 2)
Map.addLayer(image, {'palette': 'red'}, "Bangalore geometry")
Map

## Computing the Land Surface Temperature

In [ ]:
# https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2#colab-python
# Applies scaling factors.
# def apply_scale_factors(image):
#   optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
#   thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
#   return image.addBands(optical_bands, None, True).addBands(
#       thermal_bands, None, True
#   )

In [ ]:
colThsummer = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
.filterBounds(bangGeometry) \
.filterDate('2024-01-01', '2025-11-01') \
.filter(ee.Filter.calendarRange(3, 5, 'month')) 

In [ ]:
## colThsummer

In [ ]:
def maskL8sr(image):
    ## Code obtained from:
    ## https://courses.spatialthoughts.com/end-to-end-gee-supplement.html#derive-lst-from-landsat-images 
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturationMask = image.select('QA_RADSAT').eq(0)

    # Scaling factors
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return image.addBands(opticalBands, None, True) \
    .addBands(thermalBands, None, True) \
    .updateMask(qaMask) \
    .updateMask(saturationMask)


In [ ]:
colThsummer = colThsummer.map(maskL8sr)
## colThsummer

In [ ]:
imageThsummer = colThsummer.median()
## imageThsummer

In [ ]:
image_clipped = imageThsummer.clip(bangGeometry)

visualisation = {
  'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 
  'min': 0,
  'max': 0.5,
  'gamma': [0.95, 1.1, 1]
}

Map.addLayer(image_clipped, visualisation, 'True Color Composite Bangalore')

Map

In [ ]:
thermalsummer = imageThsummer.select('ST_B10') \
.clip(bangGeometry) \

Map.addLayer(thermalsummer, {
"min": 280,
"max": 330,
"palette": ['blue', 'white', 'red']},'Landsat_BT')

Map

In [ ]:
## Convert to Celsius
thermal_summer_c = (
thermalsummer
.clip(bangGeometry)
.subtract(273.15)
.rename('LST_C')
)

In [ ]:
Map.addLayer(thermal_summer_c, {
    "min": 35,
    "max": 42,
    "palette": ['blue', 'white', 'red'],
},
'LST_Landsat')

vis_params = {'min': 35, 'max': 42, 'palette': ['blue', 'white', 'red']}
Map.add_colorbar(vis_params, label='Land Surface Temperature (°C)',position= 'bottomright')
Map

Mask the pixels for water. We use the method for water masking of Chakraborty (2023):

In [ ]:
# Generate a water mask.
water = ee.Image('JRC/GSW1_0/GlobalSurfaceWater').select(
'occurrence')
notWater = water.mask().Not()

In [ ]:
thermal_summer_c_masked = thermal_summer_c.updateMask(notWater)

In [ ]:
Map.addLayer(thermal_summer_c_masked, {
    "min": 35,
    "max": 42,
    "palette": ['blue', 'white', 'red'],
},
'LST_Landsat water masked')

vis_params = {'min': 35, 'max': 42, 'palette': ['blue', 'white', 'red']}
# Map.add_colorbar(vis_params, label='Land Surface Temperature (°C)',position= 'bottomright')
Map

## Zonal statistics for LST

In [ ]:
lstwardsmanualsummer = thermal_summer_c_masked.reduceRegions(
collection=bangalore,
reducer=ee.Reducer.mean(),
scale=30,
)

In [ ]:
## lstwardsmanualsummer

In [ ]:
lstwardsmanualsummer = lstwardsmanualsummer.select(['ward_id', 'ward_name', 'mean'])

In [ ]:
outdir = PROCESSED_DIR /'lstband10summer_bang_masked.csv'
print(outdir)

In [ ]:
geemap.ee_export_vector(lstwardsmanualsummer, outdir, verbose=True)

In [ ]:
df_lst_mean_summer = pd.read_csv(outdir)

In [ ]:
df_lst_mean_summer

In [ ]:
print(f"The minimum temperature is {df_lst_mean_summer['mean'].min()}.")
print(f"The maximum temperature is {df_lst_mean_summer['mean'].max()}.")

In [ ]:
df_wards = gpd.read_file(df_inpath)
df_wards.head()

In [ ]:
df_wards_merged_summer = df_wards.merge(df_lst_mean_summer, on = 'ward_name')

In [ ]:
df_wards_merged_summer

In [ ]:
df_wards_merged_summer = df_wards_merged_summer[['ward_id_x', 'ward_name', 'Corporatio', 'geometry', 'mean']]

In [ ]:
df_wards_merged_summer = df_wards_merged_summer.rename(columns={"ward_id_x": "ward_id"})
df_wards_merged_summer.head()

In [ ]:
outdirlst = PROCESSED_DIR / 'merged_lst_wards_summer_band10_bang_masked.shp'
print(outdirlst)

In [ ]:
df_wards_merged_summer.to_file(outdirlst)

In [ ]:
fig_path =  BASE_DIR / 'images' /'lst_ward_summer_bang_band10_masked.png'

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# IMPORTANT: capture fig and ax
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

divider = make_axes_locatable(ax)
cax = divider.append_axes("bottom", size="5%", pad=0.1)

df_wards_merged_summer.plot(
    column="mean",
    ax=ax,
    legend=True,
    cax=cax,
    legend_kwds={
        "label": "Mean Land Surface Temperature per ward (°C)",
        "orientation": "horizontal"
    },
)
plt.savefig(fig_path)
ax.set_axis_off()
plt.savefig(fig_path, dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# IMPORTANT: capture fig and ax
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

divider = make_axes_locatable(ax)
cax = divider.append_axes("bottom", size="5%", pad=0.1)

df_wards_merged_summer.plot(
    column="mean",
    ax=ax,
    legend=True,
    cax=cax,
    legend_kwds={
        "label": "Mean Land Surface Temperature per ward (°C)",
        "orientation": "horizontal"
    },
)

df_wards_merged_summer.boundary.plot(
    ax=ax,
    linewidth=0.6,
    color="black",
    alpha=0.7
)

plt.savefig(fig_path)
ax.set_axis_off()
plt.savefig(fig_path, dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
## Also add tiny labels and change the color bar a bit:
## First, need to reproject

In [ ]:
df_wards_summer_reproj = df_wards_merged_summer.to_crs(epsg=3857)

In [ ]:
fig_path =  BASE_DIR / 'images' /'lst_ward_summer_bang_band10_full_masked.png'

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# IMPORTANT: capture fig and ax
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

df_wards_summer_reproj["label_point"] = df_wards_summer_reproj.geometry.representative_point()

divider = make_axes_locatable(ax)
cax = divider.append_axes("bottom", size="5%", pad=0.1)

df_wards_summer_reproj.plot(
    column="mean",
    ax=ax,
    legend=True,
    cax=cax,
    legend_kwds={
        "label": "Mean Land Surface Temperature per ward (°C)",
        "orientation": "horizontal"
    },
)

df_wards_summer_reproj.boundary.plot(
    ax=ax,
    linewidth=0.6,
    color="black",
    alpha=0.7
)


for _, row in df_wards_summer_reproj.iterrows():
    ax.text(
        row.label_point.x,
        row.label_point.y,
        row["ward_id"],
        fontsize=8,
        ha="center",
    )

ax.set_axis_off()
plt.savefig(fig_path, dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# IMPORTANT: capture fig and ax
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

df_wards_summer_reproj["label_point"] = df_wards_summer_reproj.geometry.representative_point()

divider = make_axes_locatable(ax)
cax = divider.append_axes("bottom", size="3%", pad=0.1)

df_wards_summer_reproj.plot(
    column="mean",
    ax=ax,
    legend=True,
    cax=cax,
    legend_kwds={
        "label": "Mean Land Surface Temperature per ward (°C)",
        "orientation": "horizontal"
    },
)

df_wards_summer_reproj.boundary.plot(
    ax=ax,
    linewidth=0.6,
    color="black",
    alpha=0.7
)


for _, row in df_wards_summer_reproj.iterrows():
    ax.text(
        row.label_point.x,
        row.label_point.y,
        row["ward_id"],
        fontsize=8,
        ha="center",
    )

ax.set_axis_off()
plt.savefig(fig_path, dpi = 300, bbox_inches = "tight")
plt.show()

## References

T. Chakraborty, 2023, “Heat islands,” in Cloud-based remote sensing with Google Earth Engine, J. A. Cardille, M. A. Crowley, D. Saah and N. E. Clinton, Eds. Springer Open, [Online] Available: https://link.springer.com/book/10.1007/978-3-031-26588-4

Gandhi, Ujaval, 2021. End-to-End Google Earth Engine Course. Spatial Thoughts. https://courses.spatialthoughts.com/end-to-end-gee.html

Wu, Q. (2023). Earth Engine and Geemap. Geospatial Datascience with Python (1st ed.). Locate Press.  https://doi.org/10.21105/joss.02305 